# Workflow Orchestration — Airflow & Prefect

In [ ]:
# Day 42 — Workflow Orchestration
#
# You wrote a script to download data. You wrote another script to train a model.
# You wrote a third script to deploy it.
# How do you run them in order, every Sunday at 2 AM, and alert Slack if step 2 fails?
#
# The Old Way: 'cron' jobs. (Hard to monitor, no retry logic, no dependency management).
# The Modern Way: Data Orchestrators like Apache Airflow or Prefect.
#
# Airflow uses Python to define DAGs (Directed Acyclic Graphs). 
# A DAG is just a map of tasks: Task A -> Task B -> Task C.
#
# Today we'll cover:
#   1. What is a DAG?
#   2. Basic Airflow DAG with Python Operators
#   3. Task Dependencies (Bitshift operators)
#   4. An alternative: Prefect (Modern Pythonic Orchestration)
#
# Install: pip install apache-airflow prefect

### 1. Airflow Concepts: The DAG

In [ ]:
# A DAG (Directed Acyclic Graph) means:
# Directed: Task 1 goes to Task 2 (there is an arrow).
# Acyclic: Task 2 does NOT go back to Task 1 (no infinite loops!).
# Graph: A visual map of nodes.
#
# In Airflow, a DAG is defined as a Python object.
# Note: Airflow is a heavy server framework. We write the DAG file here, 
# but in reality, you drop this file into Airflow's /dags directory to run it.

import os
os.makedirs('airflow_demo', exist_ok=True)

airflow_dag = '''from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# Default arguments applied to all tasks
default_args = {
    \'owner\': \'devops_team\',
    \'retries\': 3,                 # If it fails, retry 3 times
    \'retry_delay\': timedelta(minutes=5), # Wait 5 minutes between retries
}

# Define the DAG
with DAG(
    dag_id=\'daily_model_training\',
    default_args=default_args,
    description=\'Downloads data and trains the ML model\',
    schedule_interval=\'0 2 * * *\',  # Cron expression: 2:00 AM every day
    start_date=datetime(2024, 1, 1),
    catchup=False  # Don\'t run past dates if we just turned it on
) as dag:
    pass # We will add tasks here next!
'''

with open('airflow_demo/ml_dag.py', 'w') as f:
    f.write(airflow_dag)
    
print("DAG shell created.")

### 2. Operators and Dependencies

In [ ]:
# Tasks are defined using "Operators".
# PythonOperator runs a Python function.
# BashOperator runs a bash command.

airflow_full = '''from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from datetime import datetime, timedelta

# --- 1. Define the actual Python functions ---
def download_data():
    print("Downloading from Azure Blob Storage...")
    # boto3 or azure sdk code goes here
    return "data_v1.csv"

def train_model():
    print("Training Scikit-Learn model...")
    # sklearn code goes here

# --- 2. Define the DAG ---
with DAG(
    dag_id=\'daily_model_training\',
    schedule_interval=\'0 2 * * *\',
    start_date=datetime(2024, 1, 1),
    catchup=False
) as dag:

    # Task 1: Download
    task_download = PythonOperator(
        task_id=\'download_data_task\',
        python_callable=download_data
    )

    # Task 2: Train
    task_train = PythonOperator(
        task_id=\'train_model_task\',
        python_callable=train_model
    )

    # Task 3: Cleanup (using bash!)
    task_cleanup = BashOperator(
        task_id=\'cleanup_temp_files\',
        bash_command=\'rm -rf /tmp/ml_data/*\'
    )

    # --- 3. Define the Dependencies (The Graph) ---
    # The bitshift operator (>>) means "runs before"
    task_download >> task_train >> task_cleanup
'''

with open('airflow_demo/ml_dag_full.py', 'w') as f:
    f.write(airflow_full)

print("Airflow DAG with dependencies created!")
print("The magic line is: task_download >> task_train >> task_cleanup")

### 3. The Modern Alternative: Prefect

In [ ]:
# Airflow was created by Airbnb in 2014. It is robust but can be clunky 
# (requires setting up databases and web servers just to test a script).
#
# Prefect is a modern alternative. You just add `@flow` and `@task` decorators
# to your normal Python code, and it magically becomes an orchestrated pipeline!

prefect_code = '''from prefect import flow, task
import random

# Define tasks with @task
@task(retries=3, retry_delay_seconds=5)
def download_data():
    print("Downloading...")
    if random.random() < 0.3:  # Simulate a random network failure
        raise ValueError("Network Error! Prefect will retry this.")
    return [1, 2, 3, 4, 5]

@task
def train_model(data):
    print(f"Training on {len(data)} rows...")
    return "Model_V1"

# Define the workflow with @flow
@flow(name="Daily ML Pipeline")
def main_pipeline():
    # Notice how pythonic this is! No weird >> operators.
    # You just call functions like normal Python.
    data = download_data()
    model = train_model(data)
    print(f"Pipeline complete. Produced: {model}")

if __name__ == "__main__":
    # You run this just like a normal python script: python prefect_demo.py
    main_pipeline()
'''

with open('airflow_demo/prefect_demo.py', 'w') as f:
    f.write(prefect_code)

print("Prefect demo created!")
print("Look how much cleaner Prefect is than Airflow. It embraces native Python.")

In [ ]:
# Practice Questions:
#
# 1. In Airflow, what does `Task_A >> [Task_B, Task_C]` mean?
# 2. Why is using an Orchestrator better than a Linux `cron` job for ML pipelines?
# 3. What is the main structural difference between how you build a DAG in Airflow vs Prefect?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The MLOps DAG
# Write a Prefect `@flow` that represents an end-to-end MLOps pipeline.
# It should have the following `@task`s:
# - `check_data_drift()`
# - `retrain_model()` (only runs if drift is detected)
# - `push_to_mlflow()`
# - `deploy_to_kubernetes()`

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Dependency Management: If downloading data fails, training shouldn't start. Orchestrators handle this natively.
# 2. Observability: Airflow and Prefect give you beautiful UI dashboards showing green (passed) and red (failed) tasks.
# 3. MLOps Standard: You cannot build an automated retraining pipeline (CT/CD) without an orchestrator.